In [3]:
#packages
import numpy as np
import matplotlib.pyplot as plt
from public_tests_w4 import *
from utils_w4 import *

%matplotlib inline

In [ ]:
#Problem Statement
''' 
Suppose you are starting a company that grows and sells wild mushrooms:
    Since not all mushrooms are edible, you'd like to be able to tell whether a given mushroom is 
     edible or poisonous based on it's physical attributes
    You have some existing data that you can use for this task.

    Can you use the data to help you identify which mushrooms can be sold safely?

Note: The dataset used is for illustrative purposes only. It is not meant to be a guide 
       on identifying edible mushrooms.
'''

In [ ]:
#dataset
# You have 10 examples of mushrooms. For each example, you have
#Three features:
#       Cap Color (Brown or Red),
#       Stalk Shape (Tapering (as in \/) or Enlarging (as in /\)), and
#       Solitary (Yes or No)
#Label
#    Edible (1 indicating yes or 0 indicating poisonous)
#for ease of implementation, we one-hot encode the dataset

In [4]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

In [5]:
#get familiar with the dataset variables
print(f'First few elements if X_train:\n', X_train[:5])
print(f'Type of X_train:', type(X_train))


First few elements if X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>


In [6]:
print(f'First few elements of y_train:\n', y_train[:5])
print(f'The type of y_train:\n', type(y_train))

First few elements of y_train:
 [1 1 0 0 1]
The type of y_train:
 <class 'numpy.ndarray'>


In [7]:
#check the shape and how many training examples are in the dataset
print(f'The shape of X_train: \n', X_train.shape)
print(f'The shape of y_train:\n', y_train.shape)
print(f'Number of training examples (m): ', len(X_train))

The shape of X_train: 
 (10, 3)
The shape of y_train:
 (10,)
Number of training examples (m):  10


In [8]:
#Decision Tree
''' 
    Calculate the entropy at a node
    Split the dataset at a node into left and right branches based on a given feature
    Calculate the information gain from splitting on a given feature
    Choose the feature that maximizes information gain
'''
#calculate entropy
def compute_entropy(y):
    ''' 
        Computes the entropy for:

        Args:
            y (ndarray) : numpy array indicating whether each example at a node is edible(1)
             or poisonous(0)
        Returns:
            entropy (float): entropy at that node
    '''
    entropy = 0
    if len(y) != 0:
        #calculate the fraction of edible examples
        p1 = len(y[y==1]) / len(y)

        #for p1=0 and p1=1, set the entropy to 0(to handle 0log0)
        if p1 != 0 and p1 != 1:
            entropy = -p1 * np.log2(p1) - (1 -p1) * np.log2(1 - p1)
        else:
            entropy = 0
    return entropy




In [9]:
#compute entropy at the root node(with all the examples)
#Since we have 5 edible and 5 non-edible mushrooms, the entropy shpuld be 1
print('Entropy at root node: ', compute_entropy(y_train))

 #unit test
compute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed. 


In [10]:
#split data set to take the left and right branches
def split_dataset(X, node_indices, feature):
    ''' 
    Splits the data at the given node into left and right branches

    Args:
        X (ndarray):  Data matrix of shape (n_samples, n_features)
        node_indices(list) : List containing the active indices. (the samples being considered at this step)
        feature (int): Index of feature to split on

    Returns:
        left_indices (list): Indices with feature value == 1
        right_indices (list): Indices with feature value == 0
        
    '''
    left_indices = []
    right_indices = []
    #go through the indices at that node
    for i in node_indices:
        #check if the value of X at that index for the feature is 1
        if X[i][feature] == 1:
            left_indices.append(i)
        else: #the value of X at that index for the feature is 0
            right_indices.append(i)
    return left_indices, right_indices


In [12]:
#check the above implementation
#Case 1
root_indices = [0,1,2,3,4,5,6,7,8,9]

#the dataset onlyhas three feature, so this value can be
#  0(Brown Cap), 1(Tapering Stalk Shape), or 2(Solitary)
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print('CASE 1:')
print('Left indices: ', left_indices)
print('Right indices: ', right_indices)

#visualize the split
#generate_split_viz(root_indices, left_indices, right_indices, feature)

print()

#Case 2

root_indices_subset = [0,2,4,6,8]
left_indices, right_indices = split_dataset(X_train, root_indices_subset, feature)

print('CASE 2:')
print('Left indices: ', left_indices)
print('Right indies: ', right_indices)

#Visualize the split
#generate_split_viz(root_indices_subset, left_indices, right_indices, feature)

#unit test
split_dataset_test(split_dataset)

CASE 1:
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]

CASE 2:
Left indices:  [0, 2, 4]
Right indies:  [6, 8]
 All tests passed.


In [13]:
#Calculate information gain --
# function takes in the training data, the indices at a node and a feature to split on 
# and returns the information gain from the split.
def compute_information_gain(X,y, node_indices, feature):
    '''
    Compute the information gain of splitting the node on a given feature

    Args:
        X (ndarray) : Data matrix of shape (n_samples, n_features)
        y (ndarray like): List or ndarray with n_samples containing the target variable
        node_indices (ndarray) : List containing the active indices.(the samples being considered in this step)
        feature (int) : Index of feature to split on
    Returns:
        cost (float): cost computed
    '''
    #split the dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)

    #variables to use
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    information_gain = 0

    #compute the entropy at the node using the `compute_entropy()`
    node_entropy = compute_entropy(y_node)

    #entropy at the left branch
    left_entropy = compute_entropy(y_left)

    #entropy at the right branch
    right_entropy = compute_entropy(y_right)

    #compute the proportion of examples at the left branch
    w_left = len(X_left) / len(X_node)

    #compute propirton of examples at the right branch
    w_right = len(X_right) / len(X_node)

    #compute the weighted entropy from the split using `w_left, w_right, left_entropy, right_entropy`
    weighted_entropy = w_left*left_entropy + w_right*right_entropy

    #calculate the information gain
    information_gain = node_entropy - weighted_entropy

    return information_gain

In [14]:
#check the above implementation
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print('Information Gain from splitting the root on brown cap feature: ', info_gain0)

info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print('Information Gain from splitting the root node on tapering stalk shape feature:', info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print('Information Gain from splitting the root on solitary feature: ', info_gain2)

#unit test
compute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap feature:  0.034851554559677034
Information Gain from splitting the root node on tapering stalk shape feature: 0.12451124978365313
Information Gain from splitting the root on solitary feature:  0.2780719051126377
 All tests passed.


In [ ]:
#observation
# Solitary feature has the highest info gain so it's the best feature to split on at the root node

In [16]:
#Get best split
def get_best_split(X, y, node_indices):
    ''' 
    Returns the optimal feature and threshold value to split the node data

    Args:
        X (ndarray): Data matrix of shape(n_samples, n_features)
        y (array like) : List containing the active indices(the samples being considered in this step)

    Returns:
        best_feature (int): The index of the best feature to split 
    '''
    num_features = X.shape[1]

    best_feature = -1

    max_info_gain = 0
    #iterate through all the features:
    for feature in range(num_features):
        #info gain from splitting this feature
        info_gain = compute_information_gain(X, y, node_indices, feature)

        #if the info gain is larger than the max seen so far
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_feature = feature
    return best_feature   

In [ ]:
#check the above implementation
best_feature = get_best_split(X_train, y_train, root_indices)
print('Best feature to split on: %d' %best_feature)

Best feature to split on: 2


In [18]:
#unit test
get_best_split_test(get_best_split)

 All tests passed.


In [19]:
#Building the decision tree - max_depth=2
from sympy import beta


tree = []
def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    ''' 
    Builds a tree using the recursive algorithm that split the dataset into 2 subgroups 
     at each node

    Args:
      X (ndarray) : Data matrix of shape (n_samples, n_features)
      y (array like) : list or ndarray with n_samples containing target variables
      node_inidces (ndarray) : list containing the active indice (samples being considered in this step)
      branch_name (string) : Name of the branch ['Root', 'Left', 'Right']
      max_depth (int): max depth of the resulting tree
      current_depth (int) : current depth. Parameter used during recursive call
    '''

    #max depth reached - stop splitting
    if current_depth == max_depth:
        formatting = ' ' * current_depth + '-' * current_depth
        print(formatting, '%s leaf node with indices' % branch_name, node_indices)
        return
    
    #otherwise get best split and split the data
    #get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices)

    formatting = '-'*current_depth
    print('%s Depth %d, %s: Split on feature: %d' % (formatting, current_depth, branch_name, best_feature))

    #split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))

    #continue splitting the left and right cild, increment current depth
    build_tree_recursive(X, y, left_indices, 'Left', max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, 'Right', max_depth, current_depth+1)
    



In [ ]:
build_tree_recursive(X_train, y_train, root_indices, 'Root', max_depth=2, current_depth=0)
generate_tree_viz(root_indices, y_train, tree)